In [ ]:
# Ajoute ces lignes pour s'assurer que le code peut charger correctement les fichiers .csv lorsqu'il est exécuté dans Google Colab.
import sys, os
if 'google.colab' in sys.modules:
    !git clone https://github.com/nickcollins-craft/USMA1Q-Methodes-Numeriques.git
    os.chdir('USMA1Q-Methodes-Numeriques/seance_5')

# USMA1Q — Méthodes Numériques
## Exercices Séance 5 — Résolution des systèmes linéaires — SOLUTIONS

## Exercice 1 — Conduction thermique 1D stationnaire

Une barre d'acier de longueur $L = 0{,}5$ m est maintenue à $T_{\text{gauche}} = 100$ °C à son extrémité gauche et à $T_{\text{droite}} = 25$ °C à son extrémité droite. En régime stationnaire sans source de chaleur interne, l'équation de la chaleur 1D est :

$$\frac{\partial^2 T}{\partial x^2} = 0$$

En discrétisant sur $n+2$ noeuds équidistants (les 2 noeuds frontières sont fixes), la discrétisation par différences finies sur les $n$ noeuds intérieurs donne :

$$T_{i-1} - 2T_i + T_{i+1} = 0 \quad \text{pour } i = 1, \ldots, n$$

Ce qui produit un système tridiagonal $K T = b$.

**1.** Construisez la matrice tridiagonale $K$ de taille $n \times n$ avec $n = 7$ en utilisant `np.diag()`.  
**2.** Construisez le vecteur $b$ en incorporant les conditions aux limites.  
**3.** Résolvez le système avec `np.linalg.solve()`. Vérifiez la solution avec `np.allclose()`.  
**4.** Tracez le profil de température complet (y compris les noeuds frontières) en fonction de la position $x$.  
**5.** La solution analytique exacte est une droite : $T(x) = T_{\text{gauche}} + (T_{\text{droite}} - T_{\text{gauche}}) \cdot x/L$. Comparez votre solution numérique à la solution analytique. Quelle est l'erreur maximale, et pourquoi est-elle si faible ?  
**6.** *(Bonus)* Ajoutez une source de chaleur uniforme $q = 5000$ W/m³ (conductivité thermique $k = 50$ W/(m·K)). La source modifie le membre droit : $b_i \mathrel{{-}{=}} q \cdot (dx)^2 / k$. Tracez la nouvelle solution et expliquez la forme de la courbe.  
_Il a été un erreur de signe ici, il a du avoir -= en place de += qui se trouve dans les exercices_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 7              # noeuds intérieurs
L = 0.5            # m — longueur de la barre
T_gauche = 100.0   # °C
T_droite  =  25.0  # °C
dx = L / (n + 1)   # pas de discrétisation

# 1. Matrice tridiagonale K avec np.diag()
# Diagonale principale : -2 ; sur- et sous-diagonales : +1
K = np.diag(-2 * np.ones(n), k=0) + np.diag(np.ones(n - 1), k=1) + np.diag(np.ones(n - 1), k=-1)
print("K =", K)

In [ ]:
# 2. Membre droit
# b est nul partout sauf aux noeuds adjacents aux frontières
b = np.zeros(n)
b[0]  = -T_gauche  # condition gauche T_0   = 100 °C
b[-1] = -T_droite  # condition droite T_n+1 =  25 °C
print("b = ", b)

In [ ]:
# 3. Résolution
T_int = np.linalg.solve(K, b)
print("Températures intérieures (°C) : ", np.round(T_int, 4))

# Vérification : K @ T_int doit être (presque) égal à b
print("Vérification K @ T = b : ", np.allclose(K @ T_int, b))

In [ ]:
# 4. Profil complet : noeuds frontières inclus
x_noeuds  = np.linspace(0, L, n + 2)
T_complet = np.concatenate([[T_gauche], T_int, [T_droite]])

# 5. Solution analytique : droite reliant les deux températures imposées
T_analytique = T_gauche + (T_droite - T_gauche) * x_noeuds / L

plt.figure(figsize=(8, 5))
plt.plot(x_noeuds, T_complet, 'o-', color='steelblue', linewidth=2, label='Solution numérique')
plt.plot(x_noeuds, T_analytique, '--', color='tomato', linewidth=2, label='Solution analytique')
plt.xlabel("Position x (m)", fontsize=13)
plt.ylabel("Température T (°C)", fontsize=13)
plt.title("Conduction thermique 1D — profil de température", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

err_max = np.max(np.abs(T_complet - T_analytique))
print("Erreur maximale numérique vs analytique :", err_max, "°C")

**5. Pourquoi l'erreur maximale est si faible ?**

L'équation de la chaleur 1D stationnaire sans terme source ($\partial^2 T / \partial x^2 = 0$) admet une solution analytique exactement linéaire : $T(x) = T_{\text{gauche}} + (T_{\text{droite}} - T_{\text{gauche}}) \cdot x/L$.

La discrétisation par différences finies centrées du second ordre peut reproduire exactement les courbes linéaires, et donc la discrétisation est exacte et la seule erreur est d'ordre machine ($\sim 10^{-14}$ °C), due à l'arithmétique en virgule flottante.

In [ ]:
# 6. Bonus — avec source de chaleur uniforme q (W/m³)
k_therm = 50.0    # W/(m·K) — conductivité thermique de l'acier
q       = 5000.0  # W/m³ — densité volumique de source

# Le terme source s'ajoute uniformément au membre droit : Δb_i = q * dx² / k
b_source = np.zeros(n)
b_source[0]  = -T_gauche
b_source[-1] = -T_droite
b_source -= q * dx**2 / k_therm   # contribution uniforme de la source

T_source         = np.linalg.solve(K, b_source)
T_source_complet = np.concatenate([[T_gauche], T_source, [T_droite]])

plt.figure(figsize=(8, 5))
plt.plot(x_noeuds, T_complet, 'o-', color='steelblue', linewidth=2, label='Sans source')
plt.plot(x_noeuds, T_source_complet, 's-', color='tomato', linewidth=2, label='Avec source q = 5 000 W/m³')
plt.xlabel("Position x (m)", fontsize=13)
plt.ylabel("Température T (°C)", fontsize=13)
plt.title("Conduction thermique 1D — effet de la source de chaleur", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

**6. Pourquoi la courbe a une telle forme ?**

Avec une source de chaleur uniforme $q$, l'équation de la chaleur devient $\partial^2 T / \partial x^2 = -q/k$. La solution analytique est **parabolique** :

$$T(x) = -\frac{q}{2k}x^2 + C_1 x + C_2$$

La source génère de la chaleur uniformément → elle élève les températures intérieures par rapport à la solution sans source (droite). La courbe est donc **au-dessus** de la droite dans tout le domaine. Le vertex de la parabole se trouve à $x^* = -C_1 / (q/k)$, ce qui pour ces conditions aux limites donne $x^* = -1{,}25$ m, hors du domaine $[0, 0{,}5]$ m : la température **décroît de façon monotone** de gauche à droite, sans maximum intérieur. La deuxième dérivée $\partial^2 T / \partial x^2 = -q/k < 0$ indique une parabole **concave vers le bas** (ouverture vers le bas), dont la pente de décroissance s'accentue vers la droite — d'où la forme légèrement courbée par rapport à la droite.

## Exercice 2 — Conditionnement et stabilité numérique

Le **numéro de condition** $\kappa(A) = \|A\| \cdot \|A^{-1}\|$ quantifie la sensibilité de la solution d'un système $Ax = b$ aux perturbations : une perturbation relative $\|\delta b\|/\|b\|$ sur le membre droit peut amplifier l'erreur relative sur la solution d'un facteur allant jusqu'à $\kappa(A)$.

**1.** Construisez la matrice $A_{\text{bon}} = \begin{pmatrix} 3 & 1 \\ 1 & 4 \end{pmatrix}$. Calculez son numéro de condition avec `np.linalg.cond()`. Combien de chiffres significatifs sont perdus ?

**2.** La **matrice de Hilbert** $H_n$ est définie par $H_{ij} = 1/(i+j-1)$ pour $i,j=1,2,3...$. Écrivez une fonction `hilbert(n)` qui la construit pour n'importe quelle taille $n$. Calculez $\kappa(H_6)$ et estimez le nombre de chiffres significatifs perdus sur les 15 disponibles en `float64`.

**3.** Posez $b = [1, 1, 1, 1, 1, 1]^\top$ et résolvez $H_6 x = b$ avec `np.linalg.solve()`. Calculez le **résidu** $r = b - H_6 x$ et sa norme. Un résidu faible garantit-il une solution précise ?

**4.** Perturbez légèrement le membre droit : $b' = b + \delta b$ avec $\delta b = 0{,}001 \cdot \mathbf{1}$ (perturbation de 0,1 %). Comparez la variation relative de la solution $\|\Delta x\|/\|x\|$ à la borne théorique $\kappa \cdot \|\delta b\|/\|b\|$.

**5.** Calculez $\kappa(H_n)$ pour $n \in \{4, 6, 8, 10, 12\}$ et affichez le nombre de chiffres perdus. Comment évolue le conditionnement avec la taille ?

In [ ]:
import numpy as np

# 1. Matrice bien conditionnée
A_bon = np.array([[3.0, 1.0],
                  [1.0, 4.0]], dtype=float)
print("cond(A_bon) =", round(np.linalg.cond(A_bon), 2))
print(int(np.log10(np.linalg.cond(A_bon))), "chiffre(s) perdus sur 15")

In [ ]:
# 2. Matrice de Hilbert n×n
# H[i, j] = 1/(i + j - 1) pour i, j = 1, …, n  →  en indices Python : 1/(i + j + 1)
def hilbert(n):
    H = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            H[i, j] = 1.0 / (i + j + 1)
    return H

H6 = hilbert(6)
kappa = np.linalg.cond(H6)
chiffres_perdus = int(np.log10(kappa))
print("cond(H6) =", round(kappa, 2))
print("Chiffres perdus (estimé) :", chiffres_perdus, "sur", 15)
print("Chiffres significatifs restants :", max(0, 15 - chiffres_perdus))

In [ ]:
# 3. Résolution et résidu
b    = np.ones(6)
x_H6 = np.linalg.solve(H6, b)
r    = b - H6 @ x_H6

print("x =", np.round(x_H6, 4))
print("résidu ||r|| =", round(np.linalg.norm(r), 10))
print("résidu relatif ||r||/||b|| =", round(np.linalg.norm(r) / np.linalg.norm(b), 10))

In [ ]:
# 4. Perturbation de b : δb = 0.001 × 1_n (perturbation de 0.1 %)
delta_b = 0.001 * np.ones(6)
b_pert  = b + delta_b
x_pert  = np.linalg.solve(H6, b_pert)

# Variation relative de la solution
delta_x_rel = np.linalg.norm(x_pert - x_H6) / np.linalg.norm(x_H6)
# Perturbation relative de b
delta_b_rel = np.linalg.norm(delta_b) / np.linalg.norm(b)
# Borne supérieure théorique : κ × ||δb||/||b||
amplification_theorique = kappa * delta_b_rel

print("||Δx||/||x|| (observé)       =", round(delta_x_rel, 6))
print("κ × ||Δb||/||b|| (théorique) =", round(amplification_theorique, 6))
print("→ L'amplification observée est ≤ la borne théorique.")

In [ ]:
# 5. Évolution du numéro de condition avec la taille
print("n   | cond(Hn)          | chiffres perdus")
print("----|-------------------|----------------")
for n_h in [4, 6, 8, 10, 12]:
    Hn   = hilbert(n_h)
    c    = np.linalg.cond(Hn)
    perd = int(np.log10(c))
    print(n_h, " | ", round(c, 3), " | ", min(perd, 15))

## Exercice 3 — Moindres carrés : loi de fluage

On a réalisé un essai de fluage sur un alliage d'aluminium à 150 °C.  
Les déformations mesurées à différents instants sont données ci-dessous.

| Temps (h) | Déformation (%) |
|-----------|-----------------|
| 1  | 0.12 |
| 5  | 0.21 |
| 10 | 0.28 |
| 50 | 0.50 |
| 100 | 0.66 |
| 500 | 1.18 |
| 1000 | 1.56 |
| 5000 | 2.80 |

On suppose une loi puissance en temps : $\varepsilon = A \cdot t^n$

**1.** Linéarisez la loi par passage au logarithme.  
**2.** Construisez la matrice $A$ du système linéarisé et résolvez via `np.linalg.lstsq()`. Reportez les valeurs de $A$ et $n$.  
**3.** Comparez avec `np.polyfit(np.log(t), np.log(eps), deg=1)`. Les résultats sont-ils identiques ?  
**4.** Tracez les données et la courbe ajustée sur un graphique log-log.  
**5.** Tracez les résidus (données − modèle) sur une échelle linéaire. Le modèle est-il satisfaisant ?  
**6.** *(Bonus)* Calculez le coefficient de détermination $R^2$ de l'ajustement dans l'espace logarithmique.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Données de l'essai de fluage
t   = np.array([1, 5, 10, 50, 100, 500, 1000, 5000], dtype=float)  # heures
eps = np.array([0.12, 0.21, 0.28, 0.50, 0.66, 1.18, 1.56, 2.80])   # %

# 1. Linéarisation : ln(ε) = n·ln(t) + ln(A)
ln_t   = np.log(t)
ln_eps = np.log(eps)

# 2. Matrice du système linéarisé [ln(t), 1] @ [n, ln(A)] = ln(ε)
A_mat = np.column_stack([ln_t, np.ones_like(ln_t)])
coeffs, res_sq, rang, sv = np.linalg.lstsq(A_mat, ln_eps, rcond=None)

n_ajuste = coeffs[0]          # exposant
A_ajuste = np.exp(coeffs[1])  # préfacteur

print("Loi de fluage ajustée : eps = A * t^n")
print("A =", round(A_ajuste, 5), " (unité : %/h^n)")
print("n =", round(n_ajuste, 4))

In [ ]:
# 3. Comparaison avec np.polyfit
# np.polyfit(x, y, deg=1) → [pente, ordonnée à l'origine]
coeffs_poly = np.polyfit(ln_t, ln_eps, deg=1)
n_poly = coeffs_poly[0]
A_poly = np.exp(coeffs_poly[1])

print("np.polyfit → A =", round(A_poly, 5), ", n =", round(n_poly, 4))
print("lstsq     → A =", round(A_ajuste, 5), ", n =", round(n_ajuste, 4))
print("Identiques :", np.allclose([n_poly, A_poly], [n_ajuste, A_ajuste], rtol=1e-6))
# Les deux méthodes sont algébriquement équivalentes pour un ajustement linéaire sans pondération,
# elles donnent exactement le même résultat.

In [ ]:
# 4. Graphique log-log
t_courbe   = np.linspace(t.min(), t.max(), 300)
eps_courbe = A_ajuste * t_courbe**n_ajuste

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].loglog(t, eps, 'o', color='steelblue', markersize=7, label='Données')
axes[0].loglog(t_courbe, eps_courbe, '-', color='tomato', linewidth=2, label=u'ε = %.4f · t^%.4f' % (A_ajuste, n_ajuste))
axes[0].set_xlabel("Temps (h)", fontsize=12)
axes[0].set_ylabel(u"Déformation (%)", fontsize=12)
axes[0].set_title(u"Loi de fluage — échelle log-log", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, which='both', alpha=0.3)

# 5. Résidus (données − modèle) sur les points mesurés
eps_ajuste_pts = A_ajuste * t**n_ajuste
residus        = eps - eps_ajuste_pts

axes[1].semilogx(t, residus, 'o', color='steelblue', markersize=7)
axes[1].axhline(0, color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel("Temps (h)", fontsize=12)
axes[1].set_ylabel(u"Résidu (données − modèle) (%)", fontsize=12)
axes[1].set_title(u"Analyse des résidus", fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Bonus — coefficient R² dans l'espace logarithmique
ln_eps_predit = np.log(A_ajuste) + n_ajuste * ln_t
ss_tot = np.sum((ln_eps - np.mean(ln_eps))**2)
ss_res = np.sum((ln_eps - ln_eps_predit)**2)
R2     = 1.0 - ss_res / ss_tot
print("R² (espace log) =", round(R2, 6))